In [2]:
import os
import sys
import pandas as pd


In [3]:
%pwd

'a:\\Future\\MLOPS\\Projects\\Data science pro\\research'

In [4]:
os.chdir("../")

In [5]:
%pwd

'a:\\Future\\MLOPS\\Projects\\Data science pro'

In [6]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelTrainerConfig:
    root_dir: Path
    train_data_path: Path
    test_data_path: Path
    model_name: str
    alpha: float
    l1_ratio: float
    target_column: str
    

    

In [7]:
%pwd

'a:\\Future\\MLOPS\\Projects\\Data science pro'

In [8]:
from src.DataSciencePro.constants import *
from src.DataSciencePro.utils.common import *

class ConfigurationManager:
    def __init__(
        self,
        config_filepath= CONFIG_FILE_PATH,
        params_filepath= PARAM_FILE_PATH,
        schema_filepath= SCHEME_FILE_PATH):
        
        self.config= read_yaml(config_filepath)
        self.params= read_yaml(params_filepath)
        self.schema= read_yaml(schema_filepath)
        
        create_directories([self.config.artifacts_root])
    
    def get_modeltrainer_config(self)-> ModelTrainerConfig:
        config= self.config.model_trainer
        params= self.params.ElasticNet
        schema= self.schema.TARGET_COLUMN

        
        
        create_directories([config.root_dir])
        
        modeltrainer_config= ModelTrainerConfig(
            root_dir= config.root_dir,
            train_data_path= config.train_data_path,
            test_data_path= config.test_data_path,
            model_name= config.model_name,
            alpha= params.alpha,
            l1_ratio= params.l1_ratio,
            target_column= schema.name
                    
        )
        return modeltrainer_config
    
    
    
    

In [9]:
import pandas as pd
from src.DataSciencePro import logger
from sklearn.linear_model import ElasticNet
import joblib

In [10]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config= config
        
    
    def train(self):
        
        train_data= pd.read_csv(self.config.train_data_path)
        test_data= pd.read_csv(self.config.test_data_path)
        
        train_x= train_data.drop([self.config.target_column], axis=1)
        test_x = train_data.drop([self.config.target_column], axis= 1)
        
        train_y= train_data[[self.config.target_column]]
        test_y = test_data[[self.config.target_column]]
        
        
        lr= ElasticNet(alpha= self.config.alpha, l1_ratio= self.config.l1_ratio, random_state= 42)
        lr.fit(train_x, train_y)
        
        
        joblib.dump(lr, os.path.join(self.config.root_dir, self.config.model_name))
        

In [11]:
try:
    config = ConfigurationManager()
    model_trainer_config= config.get_modeltrainer_config()
    model_trainer= ModelTrainer(config = model_trainer_config)
    model_trainer.train()
except Exception as e:
    raise e

[2026-05-05 14:06:05,461 - DatascienceLogger - INFO - yaml file: config\config.yaml is loaded sucessfully.]
[2026-05-05 14:06:05,465 - DatascienceLogger - INFO - yaml file: params.yaml is loaded sucessfully.]
[2026-05-05 14:06:05,486 - DatascienceLogger - INFO - yaml file: schema.yaml is loaded sucessfully.]
Created directory at: artifacts
Created directory at: artifacts/model_trainer
